In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from study_config import *

PROC = Path("../data/processed")
FIG = Path("../outputs/figures")
TAB = Path("../outputs/tables")

checks = []
def check(q, name, fn, expected, tol=0.0):
    try:
        actual = fn()
        status = "PASS" if abs(actual - expected) <= tol else "FAIL"
    except Exception:
        actual, status = None, "FILE MISSING"
    checks.append({"question": q, "check": name, "expected": expected, "actual": actual, "status": status})

def check_true(q, name, fn):
    try:
        ok = bool(fn()); status = "PASS" if ok else "FAIL"
    except Exception:
        ok, status = None, "FILE MISSING"
    checks.append({"question": q, "check": name, "expected": True, "actual": ok, "status": status})

pct = lambda x: (np.exp(x) - 1) * 100

In [2]:
# --- Question 1
check("Q1", "retail rows in analysis file", lambda: len(pd.read_csv(PROC / "retail_clean_flagged.csv")), 207601)
check("Q1", "commodity-market series", lambda: pd.read_csv(PROC / "retail_clean_flagged.csv", usecols=["market_id", "commodity_id"]).drop_duplicates().shape[0], 4589)
check("Q1", "flagged monthly changes", lambda: len(pd.read_csv(TAB / "09_flagged_jumps_all.csv")), 1721)
check("Q1", "flags in the core five (equals Step 4)", lambda: pd.read_csv(TAB / "09_flagged_jumps_all.csv")["commodity"].isin(CORE).sum(), 111)
check("Q1", "series eligible for trend/volatility", lambda: len(pd.read_csv(TAB / "09_series_stats.csv")), 2725)
check("Q1", "retail commodities in coverage table", lambda: len(pd.read_csv(TAB / "09B_commodity_coverage.csv")), 64)
check("Q1", "commodities with no eligible series", lambda: (pd.read_csv(TAB / "09B_commodity_coverage.csv")["eligible_series"] == 0).sum(), 3)
check("Q1", "markets in market profile", lambda: len(pd.read_csv(TAB / "09B_market_profile.csv")), 97)
check("Q1", "listed markets without prices", lambda: len(pd.read_csv(TAB / "09B_markets_without_prices.csv")), 35)
# --- Question 2
for k, n, m in [("pre_pan", 38, 43.9), ("pan_post", 2615, 18.8), ("pre_post", 36, 73.4)]:
    check("Q2", f"matched series {k}", lambda k=k: len(pd.read_csv(TAB / f"10_matched_series_{k}.csv")), n)
    check("Q2", f"median % change {k}", lambda k=k: pd.read_csv(TAB / f"10_matched_series_{k}.csv")["pct_chg"].median(), m, 0.05)
check("Q2", "category x region cells with data", lambda: len(pd.read_csv(TAB / "10B_category_region_cells.csv")), 68)
check("Q2", "category x region cells reliable", lambda: pd.read_csv(TAB / "10B_category_region_cells.csv")["reliable"].sum(), 61)
for k, n, m in [("pan_post", 2615, 18.8), ("pre_pan", 38, 41.1), ("pre_post", 36, 74.3)]:
    check("Q2", f"calendar-balanced series {k}", lambda k=k: len(pd.read_csv(TAB / f"13_calendar_balanced_{k}.csv")), n)
    check("Q2", f"calendar-balanced median {k}", lambda k=k: pd.read_csv(TAB / f"13_calendar_balanced_{k}.csv")["pct_chg"].median(), m, 0.05)
check("Q2", "counterfactual series (Step 14)", lambda: len(pd.read_csv(TAB / "14_excess_by_series.csv")), 26)
check("Q2", "break tests rejected at 5% (Step 14)", lambda: (pd.read_csv(TAB / "14_break_tests.csv")["p_value"] < 0.05).sum(), 22)
check("Q2", "median excess over flat level, % (Step 14)", lambda: pct(pd.read_csv(TAB / "14_excess_by_series.csv")["ex_flat"].median()), 43.4, 0.05)
check("Q2", "median excess over trend, % (Step 14)", lambda: pct(pd.read_csv(TAB / "14_excess_by_series.csv")["ex_trend"].median()), 13.9, 0.05)
# --- Question 3
check("Q3", "forecast panel rows", lambda: len(pd.read_csv(PROC / "forecast_panel.csv")), 228522)
check("Q3", "eligible series-windows", lambda: pd.read_csv(TAB / "11_eligibility_by_series_window.csv")["eligible"].sum(), 4163)
check("Q3", "SARIMA fits logged", lambda: len(pd.read_csv(TAB / "12_sarima_fit_log.csv")), 4163)
check("Q3", "SARIMA fit failures", lambda: (pd.read_csv(TAB / "12_sarima_fit_log.csv")["fail_reason"].fillna("") != "").sum(), 1)
check("Q3", "paired test observations", lambda: len(pd.read_csv(PROC / "forecast_results_paired.csv")), 49864)
check_true("Q3", "Naive beats Seasonal Naive in all 9 window-metric cells", lambda: (pd.read_csv(TAB / "12_naive_vs_snaive.csv")["verdict"] == "naive better").all())
check_true("Q3", "SARIMA beats Seasonal Naive in all 9 window-metric cells", lambda: (pd.read_csv(TAB / "12_sarima_vs_snaive.csv")["verdict"] == "sarima better").all())
check_true("Q3", "SARIMA/Seasonal Naive median error ratio within 0.72-0.86", lambda: pd.read_csv(TAB / "12_sarima_vs_snaive.csv")["median_ratio"].between(0.72, 0.86).all())
def _tr():
    t = pd.read_csv(TAB / "12B_sarima_vs_naive_tie_robust.csv")
    return (t[t["window"] == "Pandemic"]["verdict_all"] == "naive better").all() and (t[t["window"] == "Post-pandemic"]["verdict_all"] == "no clear winner").all()
check_true("Q3", "SARIMA vs Naive (tie-robust): Pandemic = Naive better, Post = no clear winner", _tr)
check_true("Q3", "drift variant worse than Naive in all windows", lambda: (pd.read_csv(TAB / "13_sarima_variants_summary.csv").query("variant == 'drift12' and vs == 'naive'")["verdict_all_metrics"] == "naive better").all())
check_true("Q3", "25-month training worse than 62-month (ratio > 1, all metrics)", lambda: (pd.read_csv(TAB / "13_post25_vs_sarima62.csv")["gm_ratio"] > 1).all())

reg = pd.DataFrame(checks)
print(reg["status"].value_counts().to_dict())
display(reg[reg["status"] != "PASS"])          # shows only problems
reg.to_csv(TAB / "15_consistency_checks.csv", index=False)

{'PASS': 38}


,question,check,expected,actual,status


In [3]:
FILES = {
 "Step 1":  [PROC/"wfp_food_prices_phl.csv", TAB/"01_validation_summary.csv"],
 "Step 2":  [PROC/"wfp_prices_tagged.csv", TAB/"02_monthly_coverage.csv", TAB/"02_period_summary.csv", FIG/"02_monthly_coverage.png"],
 "Step 3":  [PROC/"wfp_retail_tagged.csv", TAB/"03_markets_per_region_period.csv", TAB/"03_commodity_region_coverage.csv", TAB/"03_core_commodities.csv", FIG/"03_core_coverage_heatmap.png"],
 "Step 4":  [PROC/"core_retail_clean.csv", TAB/"04_flagged_jumps.csv", FIG/"04_core_price_overview.png"],
 "Step 5":  [PROC/"national_monthly_index.csv", TAB/"05_bridge_comparison.csv", FIG/"05_index_comparison.png"],
 "Step 6":  [PROC/"log_index_segments.csv", TAB/"06_volatility_by_period.csv", TAB/"06_stationarity_tests.csv", FIG/"06_monthly_log_changes.png"],
 "Step 7":  [TAB/"07_autocorr_summary.csv", TAB/"07_stl_strength.csv", FIG/"07_acf.png", FIG/"07_pacf.png", FIG/"07_stl_seasonal.png"],
 "Step 8":  [PROC/"forecasts_baselines.csv", TAB/"08_baseline_metrics.csv", TAB/"08_baseline_metrics_by_period.csv", FIG/"08_baseline_mae_by_horizon.png"],
 "Step 9":  [PROC/"retail_clean_flagged.csv", TAB/"09_flagged_jumps_all.csv", TAB/"09_flag_rate_by_commodity.csv", TAB/"09_series_stats.csv",
             TAB/"09_trend_vol_by_category.csv", TAB/"09_trend_vol_by_commodity.csv", TAB/"09_trend_vol_by_region.csv", TAB/"09_trend_vol_by_market.csv",
             TAB/"09_chain_index_category_A.csv", TAB/"09_chain_index_B.csv", FIG/"09_chain_category.png", FIG/"09_chain_region.png", FIG/"09_trend_vol_by_category.png"],
 "Step 9B": [TAB/"09B_commodity_coverage.csv", TAB/"09B_missing_commodities.csv", TAB/"09B_market_profile.csv", TAB/"09B_markets_without_prices.csv", FIG/"09B_market_map.png"],
 "Step 10": [TAB/"10_matched_series_pre_pan.csv", TAB/"10_matched_series_pan_post.csv", TAB/"10_matched_series_pre_post.csv",
             TAB/"10_pan_post_by_category.csv", TAB/"10_pan_post_by_region.csv", TAB/"10_pan_post_by_commodity.csv", TAB/"10_pan_post_by_market.csv",
             TAB/"10_three_period_profile.csv", FIG/"10_pan_post_by_category.png", FIG/"10_pan_post_by_region.png", FIG/"10_three_period_profile.png"],
 "Step 10B":[TAB/"10B_category_region_cells.csv", TAB/"10B_category_region_matrix.csv", TAB/"10B_pre_pan_by_region_category.csv", TAB/"10B_pre_post_by_region_category.csv", FIG/"10B_category_region_matrix.png"],
 "Step 11": [TAB/"11_window_design.csv", TAB/"11_eligibility_by_series_window.csv", PROC/"forecast_panel.csv", FIG/"11_window_design.png", FIG/"11_eligible_by_category.png"],
 "Step 12": [TAB/"12_sarima_fit_log.csv", TAB/"12_series_metrics.csv", TAB/"12_accuracy_by_window.csv", TAB/"12_accuracy_by_horizon.csv", TAB/"12_accuracy_by_category.csv",
             TAB/"12_accuracy_core_five.csv", TAB/"12_sarima_vs_snaive.csv", TAB/"12_sarima_vs_naive.csv", TAB/"12_naive_vs_snaive.csv",
             TAB/"12B_sarima_vs_naive_tie_robust.csv", PROC/"forecast_results_paired.csv", FIG/"12_accuracy.png"],
 "Step 13": [TAB/"13_no_onions_naive_vs_snaive.csv", TAB/"13_no_onions_sarima_vs_snaive.csv", TAB/"13_no_onions_sarima_vs_naive.csv",
             TAB/"13_by_category_naive_vs_snaive.csv", TAB/"13_by_category_sarima_vs_naive.csv", TAB/"13_sarima_variants_summary.csv", TAB/"13_post25_vs_sarima62.csv",
             TAB/"13_calendar_balanced_pan_post.csv", TAB/"13_calendar_balanced_pre_pan.csv", TAB/"13_calendar_balanced_pre_post.csv"],
 "Step 14": [TAB/"14_counterfactual_observations.csv", TAB/"14_excess_by_series.csv", TAB/"14_excess_by_commodity.csv", TAB/"14_excess_by_market.csv",
             TAB/"14_break_tests.csv", FIG/"14_counterfactual_examples.png", FIG/"14_excess_by_commodity.png"],
}
inv = pd.DataFrame([{"step": s, "file": p.name, "exists": p.exists()} for s, ps in FILES.items() for p in ps])
print("Files expected:", len(inv), "| present:", int(inv["exists"].sum()))
print("Missing:"); display(inv[~inv["exists"]])
inv.to_csv(TAB / "15_output_inventory.csv", index=False)
print("SARIMA result folders:", {f.name: f.exists() for f in [PROC/"sarima", PROC/"sarima_post25", PROC/"sarima_wide27", PROC/"sarima_drift12"]})

Files expected: 98 | present: 98
Missing:


,step,file,exists


SARIMA result folders: {'sarima': True, 'sarima_post25': True, 'sarima_wide27': True, 'sarima_drift12': True}


In [4]:
reg_rows = []
def item(q, metric, fn, src):
    try:
        v = fn()
    except Exception:
        v = "n/a (file missing)"
    reg_rows.append({"question": q, "result": metric, "value": v, "source": src})

rf = lambda: pd.read_csv(PROC / "retail_clean_flagged.csv")
item("Q1", "Retail commodities / markets / regions / categories", lambda: (lambda x: f"{x.commodity.nunique()} / {x.market_id.nunique()} / {x.admin1.nunique()} / {x.category.nunique()}")(rf()), "retail_clean_flagged.csv")
item("Q1", "Series with trend and volatility estimates", lambda: f"{len(pd.read_csv(TAB / '09_series_stats.csv')):,}", "09_series_stats.csv")
item("Q1", "Median annualized trend by category (% per year)", lambda: "; ".join(f"{r.category} {r.trend_pct_yr:.2f}" for r in pd.read_csv(TAB / "09_trend_vol_by_category.csv").itertuples()), "09_trend_vol_by_category.csv")
item("Q1", "Median monthly volatility by category (%)", lambda: "; ".join(f"{r.category} {r.vol_pct:.2f}" for r in pd.read_csv(TAB / "09_trend_vol_by_category.csv").itertuples()), "09_trend_vol_by_category.csv")
item("Q1", "Flagged monthly changes, and share in vegetables and fruits",
     lambda: (lambda f: f"{len(f):,}; {(f.category == 'vegetables and fruits').mean() * 100:.1f}%")(pd.read_csv(TAB / "09_flagged_jumps_all.csv")), "09_flagged_jumps_all.csv")
item("Q1", "Chain index at June 2026, May 2020 = 100",
     lambda: (lambda x: "; ".join(f"{c.replace('cat: ', '')} {x[c].iloc[-1]:.1f}" for c in x.columns if c.startswith("cat: ")))(pd.read_csv(TAB / "09_chain_index_B.csv")), "09_chain_index_B.csv")
item("Q2", "Pandemic -> Post-pandemic median % change, matched series",
     lambda: (lambda m: f"{m.pct_chg.median():.1f}% ({len(m):,} series, {m.market_id.nunique()} markets)")(pd.read_csv(TAB / "10_matched_series_pan_post.csv")), "10_matched_series_pan_post.csv")
item("Q2", "Pandemic -> Post-pandemic by category: median [95% CI]",
     lambda: "; ".join(f"{r['category']} {r['median_pct']:.1f} [{r['ci_low']:.1f}, {r['ci_high']:.1f}]" for _, r in pd.read_csv(TAB / "10_pan_post_by_category.csv").iterrows()), "10_pan_post_by_category.csv")
item("Q2", "Pre-pandemic comparisons (3 markets): Pre -> Pandemic / Pre -> Post median %",
     lambda: f"{pd.read_csv(TAB / '10_matched_series_pre_pan.csv').pct_chg.median():.1f} / {pd.read_csv(TAB / '10_matched_series_pre_post.csv').pct_chg.median():.1f}", "10_matched_series_pre_*.csv")
def _bal():
    a = pd.read_csv(TAB / "13_calendar_balanced_pan_post.csv").groupby("category")["pct_chg"].median()
    b = pd.read_csv(TAB / "10_pan_post_by_category.csv").set_index("category")["median_pct"]
    return f"{(a - b).abs().max():.2f}"
item("Q2", "Largest category difference, calendar-balanced vs Step 10 (percentage points)", _bal, "13_calendar_balanced_pan_post.csv")
def _cells():
    c = pd.read_csv(TAB / "10B_category_region_cells.csv"); c = c[c["reliable"]].sort_values("median_pct")
    lo, hi = c.iloc[0], c.iloc[-1]
    return f"lowest {lo.admin1} / {lo.category} {lo.median_pct:.1f}%; highest {hi.admin1} / {hi.category} {hi.median_pct:.1f}%"
item("Q2", "Lowest and highest reliable category-region cells", _cells, "10B_category_region_cells.csv")
item("Q2", "Pandemic excess over trend / flat level, median % (3 markets, Step 14)",
     lambda: (lambda w: f"{pct(w.ex_trend.median()):.1f} / {pct(w.ex_flat.median()):.1f}")(pd.read_csv(TAB / "14_excess_by_series.csv")), "14_excess_by_series.csv")
item("Q3", "Pooled MAPE (%) by window: Seasonal Naive / Naive / SARIMA",
     lambda: "; ".join(f"{w}: " + " / ".join(f"{g.set_index('model').loc[m, 'pooled_MAPE']:.2f}" for m in ["snaive", "naive", "sarima"])
                       for w, g in pd.read_csv(TAB / "12_accuracy_by_window.csv").groupby("window")), "12_accuracy_by_window.csv")
item("Q3", "SARIMA vs Seasonal Naive (all three metrics agree?)",
     lambda: "; ".join(f"{w}: {v}" for w, v in pd.read_csv(TAB / "12_sarima_vs_snaive.csv").groupby("window")["verdict"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else "mixed").items()), "12_sarima_vs_snaive.csv")
item("Q3", "SARIMA vs Naive, tie-robust (all three metrics agree?)",
     lambda: "; ".join(f"{w}: {v}" for w, v in pd.read_csv(TAB / "12B_sarima_vs_naive_tie_robust.csv").groupby("window")["verdict_all"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else "mixed").items()), "12B_sarima_vs_naive_tie_robust.csv")
item("Q4", "Dashboard", lambda: "pending (Steps 16 to 18)", "-")
results = pd.DataFrame(reg_rows)
pd.set_option("display.max_colwidth", 200)
display(results)
results.to_csv(TAB / "15_results_register.csv", index=False)

,question,result,value,source
0,Q1,Retail commodities / markets / regions / categories,64 / 97 / 17 / 4,retail_clean_flagged.csv
1,Q1,Series with trend and volatility estimates,"2,725",09_series_stats.csv
2,Q1,Median annualized trend by category (% per year),"cereals and tubers 5.01; meat, fish and eggs 5.60; pulses and nuts 3.45; vegetables and fruits 7.05",09_trend_vol_by_category.csv
3,Q1,Median monthly volatility by category (%),"cereals and tubers 3.98; meat, fish and eggs 4.08; pulses and nuts 3.96; vegetables and fruits 12.96",09_trend_vol_by_category.csv
4,Q1,"Flagged monthly changes, and share in vegetables and fruits","1,721; 95.8%",09_flagged_jumps_all.csv
5,Q1,"Chain index at June 2026, May 2020 = 100","cereals and tubers 131.3; meat, fish and eggs 140.4; pulses and nuts 130.9; vegetables and fruits 134.9",09_chain_index_B.csv
6,Q2,"Pandemic -> Post-pandemic median % change, matched series","18.8% (2,615 series, 83 markets)",10_matched_series_pan_post.csv
7,Q2,Pandemic -> Post-pandemic by category: median [95% CI],"cereals and tubers 21.4 [20.4, 23.1]; meat, fish and eggs 16.0 [15.2, 16.7]; pulses and nuts 11.2 [8.5, 13.0]; vegetables and fruits 22.6 [21.2, 24.7]",10_pan_post_by_category.csv
8,Q2,Pre-pandemic comparisons (3 markets): Pre -> Pandemic / Pre -> Post median %,43.9 / 73.4,10_matched_series_pre_*.csv
9,Q2,"Largest category difference, calendar-balanced vs Step 10 (percentage points)",0.12,13_calendar_balanced_pan_post.csv


In [5]:
TRACE = [
 ("Q1", "Data inventory: 66 commodities in file, 64 retail, 4 categories, 97 markets, 17 regions", "1, 9", ["01_validation_summary.csv", "retail_clean_flagged.csv"], []),
 ("Q1", "Coverage over time, structural gaps, period definitions", "2, 3", ["02_monthly_coverage.csv", "02_period_summary.csv", "03_markets_per_region_period.csv"], ["02_monthly_coverage.png"]),
 ("Q1", "Trends and fluctuations by commodity, category, region, market", "9", ["09_series_stats.csv", "09_trend_vol_by_category.csv", "09_trend_vol_by_commodity.csv", "09_trend_vol_by_region.csv", "09_trend_vol_by_market.csv"], ["09_trend_vol_by_category.png"]),
 ("Q1", "Chain-linked indexes by category and region", "9", ["09_chain_index_category_A.csv", "09_chain_index_B.csv"], ["09_chain_category.png", "09_chain_region.png"]),
 ("Q1", "Price anomalies (flagged, not removed)", "4, 9", ["04_flagged_jumps.csv", "09_flagged_jumps_all.csv", "09_flag_rate_by_commodity.csv"], []),
 ("Q1", "Commodity coverage and short-lived commodities; market profile and map", "9B", ["09B_commodity_coverage.csv", "09B_missing_commodities.csv", "09B_market_profile.csv", "09B_markets_without_prices.csv"], ["09B_market_map.png"]),
 ("Q1", "Volatility and stationarity, seasonality (national index, core five)", "5, 6, 7", ["national_monthly_index.csv", "06_volatility_by_period.csv", "06_stationarity_tests.csv", "07_autocorr_summary.csv", "07_stl_strength.csv"], ["05_index_comparison.png", "07_stl_seasonal.png"]),
 ("Q2", "Matched-series price changes between periods", "10", ["10_matched_series_pre_pan.csv", "10_matched_series_pan_post.csv", "10_matched_series_pre_post.csv"], []),
 ("Q2", "Differences by category, region, commodity, market", "10", ["10_pan_post_by_category.csv", "10_pan_post_by_region.csv", "10_pan_post_by_commodity.csv", "10_pan_post_by_market.csv", "10_three_period_profile.csv"], ["10_pan_post_by_category.png", "10_pan_post_by_region.png", "10_three_period_profile.png"]),
 ("Q2", "Category x region matrix; pre-pandemic regions", "10B", ["10B_category_region_cells.csv", "10B_category_region_matrix.csv", "10B_pre_pan_by_region_category.csv", "10B_pre_post_by_region_category.csv"], ["10B_category_region_matrix.png"]),
 ("Q2", "Seasonality check on the period comparisons", "13", ["13_calendar_balanced_pan_post.csv", "13_calendar_balanced_pre_pan.csv", "13_calendar_balanced_pre_post.csv"], []),
 ("Q2", "Counterfactual and structural-break check (supplementary, 3 markets)", "14", ["14_excess_by_series.csv", "14_excess_by_commodity.csv", "14_excess_by_market.csv", "14_break_tests.csv"], ["14_counterfactual_examples.png", "14_excess_by_commodity.png"]),
 ("Q3", "Window design, eligibility rules, forecast panel", "11", ["11_window_design.csv", "11_eligibility_by_series_window.csv", "forecast_panel.csv"], ["11_window_design.png", "11_eligible_by_category.png"]),
 ("Q3", "Seasonal Naive and SARIMA accuracy (MAE, RMSE, MAPE)", "12", ["12_series_metrics.csv", "12_accuracy_by_window.csv", "12_accuracy_by_horizon.csv", "12_accuracy_by_category.csv", "12_accuracy_core_five.csv", "forecast_results_paired.csv"], ["12_accuracy.png"]),
 ("Q3", "Which model performs better: decision tables", "12, 12B", ["12_sarima_vs_snaive.csv", "12_sarima_vs_naive.csv", "12_naive_vs_snaive.csv", "12B_sarima_vs_naive_tie_robust.csv", "12_sarima_fit_log.csv"], []),
 ("Q3", "Robustness: without onions, by category, training length, wider grid, drift", "13", ["13_no_onions_sarima_vs_snaive.csv", "13_no_onions_sarima_vs_naive.csv", "13_by_category_sarima_vs_naive.csv", "13_sarima_variants_summary.csv", "13_post25_vs_sarima62.csv"], []),
 ("Q3", "Baseline groundwork (national index, drift and Naive baselines)", "8", ["forecasts_baselines.csv", "08_baseline_metrics.csv"], ["08_baseline_mae_by_horizon.png"]),
 ("Q4", "Interactive dashboard: trends, period comparisons, regional movements, forecasts", "16, 17, 18", [], []),
]
have = dict(zip(inv["file"], inv["exists"]))
trace = pd.DataFrame([{"question": q, "evidence": e, "steps": s, "tables": ", ".join(t), "figures": ", ".join(f),
                       "all_files_present": all(have.get(x, False) for x in t + f) if (t or f) else None}
                      for q, e, s, t, f in TRACE])
display(trace[["question", "evidence", "steps", "all_files_present"]])
trace.to_csv(TAB / "15_traceability.csv", index=False)

,question,evidence,steps,all_files_present
0,Q1,"Data inventory: 66 commodities in file, 64 retail, 4 categories, 97 markets, 17 regions","1, 9",True
1,Q1,"Coverage over time, structural gaps, period definitions","2, 3",True
2,Q1,"Trends and fluctuations by commodity, category, region, market",9,True
3,Q1,Chain-linked indexes by category and region,9,True
4,Q1,"Price anomalies (flagged, not removed)","4, 9",True
5,Q1,Commodity coverage and short-lived commodities; market profile and map,9B,True
6,Q1,"Volatility and stationarity, seasonality (national index, core five)","5, 6, 7",True
7,Q2,Matched-series price changes between periods,10,True
8,Q2,"Differences by category, region, commodity, market",10,True
9,Q2,Category x region matrix; pre-pandemic regions,10B,True


In [6]:
def nrows(f):
    p = TAB / f if (TAB / f).exists() else PROC / f
    return f"{len(pd.read_csv(p)):,}" if p.exists() else "n/a"

CONTRACT = [
 ("dash_prices", "Q1, Q4", "market x commodity x month (excluded months absent)", "market_id, commodity_id, date",
  "date, period, segment, admin1, admin2, market_id, market, category, commodity_id, commodity, unit, price, lr, flag_jump", "retail_clean_flagged.csv (Step 9)", nrows("retail_clean_flagged.csv")),
 ("dash_series_stats", "Q1", "market x commodity series with >= 24 months", "market_id, commodity_id",
  "category, commodity, admin1, market, n_obs, trend_pct_yr, vol_pct", "09_series_stats.csv", nrows("09_series_stats.csv")),
 ("dash_chain_index", "Q1", "group x month (category or region; segment A or B)", "level, group, segment, month",
  "index (base month = 100)", "09_chain_index_category_A.csv, 09_chain_index_B.csv", "built in Step 16"),
 ("dash_markets", "Q1, Q2, Q3", "market", "market_id",
  "market, admin1, admin2, latitude, longitude, observations, commodities, months_active, first_month, last_month, periods_present, era, eligible_series, median_trend_pct_yr, median_vol_pct", "09B_market_profile.csv", nrows("09B_market_profile.csv")),
 ("dash_commodities", "Q1", "commodity", "commodity",
  "category, unit, observations, markets, first_month, last_month, min/median/max price, obs per period, eligible_series, tier", "09B_commodity_coverage.csv", nrows("09B_commodity_coverage.csv")),
 ("dash_period_series", "Q2", "comparison x market x commodity", "comparison, market_id, commodity_id",
  "category, commodity, market, admin1, p1, p2, pct_chg, pct_chg_calendar_balanced", "10_matched_series_*.csv, 13_calendar_balanced_*.csv", "built in Step 16"),
 ("dash_period_summary", "Q2", "comparison x dimension x group (overall, category, region, commodity, market, category x region)", "comparison, dimension, group",
  "series, markets, median_pct, ci_low, ci_high, reliable", "10_pan_post_by_*.csv, 10B_category_region_cells.csv", "built in Step 16"),
 ("dash_forecast_holdout", "Q3, Q4", "window x series x test month", "window, market_id, commodity_id, ym",
  "category, commodity, market, admin1, h, price (actual), f_naive, f_snaive, f_sarima (log scale)", "forecast_results_paired.csv (Step 12)", nrows("forecast_results_paired.csv")),
 ("dash_forecast_metrics", "Q3", "window x series x model", "window, market_id, commodity_id, model",
  "category, commodity, admin1, n_test, MAE, RMSE, MAPE", "12_series_metrics.csv", nrows("12_series_metrics.csv")),
 ("dash_forecast_summary", "Q3", "window x model (x category / commodity / horizon band)", "window, model, dimension, group",
  "pooled_MAPE, median_series_MAPE, ratios and verdicts vs Seasonal Naive and Naive", "12_accuracy_*.csv, 12_sarima_vs_*.csv, 12B_*.csv", "built in Step 16"),
 ("dash_forecast_future", "Q3, Q4", "series x future month (Jul 2026 to Jun 2027), flagged indicative", "market_id, commodity_id, ym",
  "f_naive, f_snaive, f_sarima, holdout MAPE of the series, indicative = True", "NOT YET PRODUCED", "built in Step 16 (refit on data to Jun 2026)"),
 ("dash_monitoring", "Q4", "series (latest month)", "market_id, commodity_id",
  "latest_month, latest_price, change_vs_prev_month_pct, change_vs_12m_pct, flag_jump", "retail_clean_flagged.csv", "built in Step 16"),
 ("dash_about", "Q4", "static text / key-value", "key",
  "period definitions, excluded months, matched-series definition, nominal prices, market coverage changes, forecast caveats", "study_config.py, Steps 2, 3, 5, 10, 12", "written in Step 16"),
]
contract = pd.DataFrame(CONTRACT, columns=["table", "serves", "grain", "key", "columns", "source", "rows_now"])
display(contract[["table", "serves", "grain", "source", "rows_now"]])
contract.to_csv(TAB / "15_dashboard_data_contract.csv", index=False)

,table,serves,grain,source,rows_now
0,dash_prices,"Q1, Q4",market x commodity x month (excluded months absent),retail_clean_flagged.csv (Step 9),"207,601"
1,dash_series_stats,Q1,market x commodity series with >= 24 months,09_series_stats.csv,"2,725"
2,dash_chain_index,Q1,group x month (category or region; segment A or B),"09_chain_index_category_A.csv, 09_chain_index_B.csv",built in Step 16
3,dash_markets,"Q1, Q2, Q3",market,09B_market_profile.csv,97
4,dash_commodities,Q1,commodity,09B_commodity_coverage.csv,64
5,dash_period_series,Q2,comparison x market x commodity,"10_matched_series_*.csv, 13_calendar_balanced_*.csv",built in Step 16
6,dash_period_summary,Q2,"comparison x dimension x group (overall, category, region, commodity, market, category x region)","10_pan_post_by_*.csv, 10B_category_region_cells.csv",built in Step 16
7,dash_forecast_holdout,"Q3, Q4",window x series x test month,forecast_results_paired.csv (Step 12),"49,864"
8,dash_forecast_metrics,Q3,window x series x model,12_series_metrics.csv,"12,486"
9,dash_forecast_summary,Q3,window x model (x category / commodity / horizon band),"12_accuracy_*.csv, 12_sarima_vs_*.csv, 12B_*.csv",built in Step 16
